# PAUT preprocessing: single file

Run the preprocessing on **one** exported PAUT volume, inspect every step and tune the parameters.
The batch notebook (`paut_batch.ipynb`) runs exactly the same code over all files.

1. **Echo start**: every A-scan is shifted so the entrance echo sits at 0 mm (this removes the pendulum offset). The echo is the first sample with |a| ≥ threshold inside the gate. A-scans without a hit get their shift from the nearby hits (robust k-NN inverse-distance weighting in mm). The shift is applied to the original RF. This uses the colleague's own functions, so results are identical to his tool.
2. **Auto detect piece**: a gate just below the aligned entrance echo only sees the part, not the tank echo, so the A-scans that reach the threshold there define the lateral crop.
3. **Crop** to the part and in depth to −4 … 6.5 mm around the entrance echo (default; echo detected on the Hilbert envelope). Axes are rebased to 0.
4. **Mirror**: optional flips, chosen from the figures at the end.

Inputs: FocusData exports (`<id>_ch1_b1_g1_dg1.npy` + `<id>_metadata.json`) under `export_root` in `paut_config.json`.
Outputs (only when `SAVE = True`) go straight to the NAS folder (`output_root`):
`<campaign>/<acquisition>/01_Raw` and `02_Processed`.
To go through **all** volumes, use the script `produccion/paut/paut_review.py` instead. It runs the same code, and decisions made here or there are shared.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd().resolve().parents[1] if Path.cwd().name == 'paut' else Path.cwd()
sys.path.insert(0, str(REPO))
from preprocess_tools import paut
from preprocess_tools.paut import qc as pqc
from preprocess_tools.paut import storage as st

CONFIG = REPO / 'produccion' / 'paut' / 'paut_config.json'
cfg = st.load_config(CONFIG)
items, problems = st.discover(cfg)
print(f"{len(items)} exported volumes found, {len(problems)} problems")
for p in problems:
    print('  ', p)

## Choose the file
Set the campaign, acquisition and the **original** id (the export file name before `_ch`).
The DB sample name is derived from the campaign rule in the config.

In [ ]:
CAMPAIGN = 'Na_Panels'             # Na_Panels | JI | Pegaso | Paneles A | F_01
ACQUISITION = 'raw_3p5MHz_16elem'  # raw_3p5MHz_16elem | echostar_3p5MHz_16elem | ...
ORIGINAL_ID = 'Na_01_01'

item = next(i for i in items if (i.campaign, i.acquisition, i.original_id) == (CAMPAIGN, ACQUISITION, ORIGINAL_ID))
print('key       :', item.key)
print('export    :', item.npy)
print('DB sample :', item.sample)
print('decision  :', st.load_review(cfg).get(item.key, 'none yet'))

## Parameters
Start from the acquisition defaults in the config, plus any earlier review decision, and change what you need here.
Values below are the colleague's tool settings stored in their `.h5` files.

In [ ]:
params = st.item_params(cfg, item)
print(json.dumps(params, indent=1))

# Examples of changes:
# params['echo_start']['threshold_pct'] = 30
# params['echo_start']['interpolate_missing'] = False   # A-scans without a hit stay unshifted
# params['depth_range_mm'] = [0, 6]                  # the colleague's crop (bit-identical to his files)
# params['footprint'].update(threshold_pct=20, gate_mm=[0, 5])

## Raw export

In [ ]:
raw = paut.load_export(item.npy, item.metadata)
print('shape (index, scan, depth):', raw.shape, ' spacing (mm):', raw.spacing_mm,
      f' depth {raw.depth_mm[0]:.2f}…{raw.depth_mm[-1]:.2f} mm')
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
i, j = raw.shape[0] // 2, raw.shape[1] // 2
ax[0].imshow(np.abs(raw.data).max(-1), aspect='equal', cmap='viridis')
ax[0].set_title('max |a| over all depth (index × scan)')
ax[1].imshow(raw.data[i].T, aspect='auto', cmap='seismic', vmin=-100, vmax=100,
             extent=[raw.scan_mm[0], raw.scan_mm[-1], raw.depth_mm[-1], raw.depth_mm[0]])
ax[1].set(title=f'B-scan at index row {i}', xlabel='scan (mm)', ylabel='depth (mm)')
ax[2].plot(raw.depth_mm, raw.data[i, j])
ax[2].set(title=f'A-scan ({i}, {j})', xlabel='depth (mm)', ylabel='amplitude (%)')
plt.show()

## Run the preprocessing

In [ ]:
out, record, qc = paut.process(raw, params)
summary = {k: record[k] for k in ('echo_start', 'footprint', 'crop', 'mirror', 'output') if k in record}
print(json.dumps(summary, indent=1, default=float))
pqc.plot_qc(raw, out, record, qc, title=item.key)
plt.show()

## Inspect A-scans (interactive)
Click anywhere on the processed C-scan, or on either B-scan, to see the A-scan at that position.
The bottom panel shows the **original** A-scan at the same physical position, with the detected entrance echo (red), the echo-start gate (orange), the threshold, and the part kept by the depth crop (green).
The toolbar's zoom and pan work too; switch them off to click again.
Needs the `ipympl` backend (installed in the `compstlar` env).

In [ ]:
%matplotlib widget
viewer = pqc.ascan_viewer(out, raw, record, qc, title=item.key)
plt.show()

## Inspect the volume (sliders)
Sliders for depth, index and scan; clicking a view moves the others to that point. The display can be RF, |RF| or envelope.
The first viewer shows the processed volume (depth relative to the entrance echo); the second shows the original export.

In [ ]:
%matplotlib widget
pqc.slice_viewer_widget(out, record['crop']['source_start_mm']['depth'], title=f'{item.key}: processed')

In [ ]:
%matplotlib widget
pqc.slice_viewer_widget(raw, 0.0, title=f'{item.key}: original export')

## Comparison with the colleague's processed file (if it exists)
Only the 49 `Na_Panels / raw_3p5MHz` samples have one. With depth crop [0, 6] (his crop; the default is now −4 … 6.5 mm) the result must be **bit-identical**, including interpolated A-scans.
Use the same mirroring as the colleague (shown below) to also compare orientation.

In [ ]:
%matplotlib inline
REF = Path(cfg.get('reference_root', '')) / CAMPAIGN / f'{ORIGINAL_ID}_fpd_processed.h5'
if ACQUISITION == 'raw_3p5MHz_16elem' and REF.exists():
    import h5py
    with h5py.File(REF) as h:
        ref = h['amplitude'][()]
        ref_meta = json.loads(h['metadata_json'][()])
    print('reference bounds:', ref_meta['bounds'], ' ours:', record['crop']['bounds'])
    print('reference mirror:', ref_meta['mirror'])
    print('reference hits  :', ref_meta['echo_start']['hit_count'], ' ours:', record['echo_start'].get('hit_count'))
    # compare in the reference's orientation (undo our mirror, apply theirs)
    flip_i = record['mirror']['index'] != ref_meta['mirror']['horizontal']
    flip_s = record['mirror']['scan'] != ref_meta['mirror']['vertical']
    cmp, _ = paut.mirror(out, index=flip_i, scan=flip_s)
    if ref.shape == cmp.shape and 'hit' in qc.get('echo_start', {}):
        b = record['crop']['bounds']
        hit = qc['echo_start']['hit'][b['index'][0]:b['index'][1], b['scan'][0]:b['scan'][1]]
        hit = hit[::-1] if ref_meta['mirror']['horizontal'] else hit
        hit = hit[:, ::-1] if ref_meta['mirror']['vertical'] else hit
        d = np.abs(cmp.data - ref).max(-1)
        same = bool((d == 0).all())
        print('BIT-IDENTICAL to the colleague output' if same else
              f"DIFFERENT: {int((d > 0).sum())} of {d.size} A-scans differ (max {d.max():.3g} %); "
              f"hit A-scans differing {int((d[hit] > 0).sum())}, interpolated {int((d[~hit] > 0).sum())}")
        pqc.plot_compare(cmp, ref, hit, title=f'{item.key} vs colleague (in the colleague orientation)')
        plt.show()
    else:
        print(f'shapes differ (ours {cmp.shape}, reference {ref.shape}): no voxel-wise comparison')
else:
    print('no reference file for this sample')

## Orientation
The four mirror combinations of the processed C-scan. Pick the one that matches the part's reference orientation.
For Na samples, compare with the XCT top view. The colleague mirrored both axes for Na_02_05, Na_03_05, Na_04_05 and Na_07_05.

In [ ]:
pqc.plot_mirror_options(out, title=f'{item.key}: mirror options (relative to the current setting)')
plt.show()

In [ ]:
MIRROR = {'index': False, 'scan': False}   # your choice

## Save
With `SAVE = True`, the result is reprocessed with `MIRROR`, then written:
- the raw TIFF goes next to the export, and is uploaded to **01_Raw** on the NAS if it isn't there yet;
- **02_Processed** gets the TIFF, `processing.json`, `processing.txt` (everything done, readable); the echo-start maps (`.npz`) stay locally next to the export (the NAS holds only TIFF volumes);
- your decision goes to `review.json`.

An existing processed file is replaced only with `OVERWRITE = True`.

In [ ]:
SAVE = False
APPROVE = True       # False stores the decision as rejected (nothing goes to 02_Processed)
OVERWRITE = False
NOTE = ''

if SAVE:
    params['mirror'] = MIRROR
    print('raw TIFF    :', st.convert_export(item))
    if APPROVE:
        result = st.run_processing(cfg, item, params)
        print('02_Processed:', st.save_processed(cfg, item, result, params, NOTE, overwrite=OVERWRITE))
        print((Path(cfg['output_root']) / item.rel('processed', '_processing.txt')).read_text())
    else:
        print('decision    :', st.reject(cfg, item, params, NOTE))